# Fiona Treatment Example

This notebook tests `shrecc.treatment_fiona` from the hourly TYNDP pickle files. It builds `Z_gross`, computes the hourly consumption mix, and checks that each consumer-country/hour mix sums to one.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from shrecc.treatment_fiona import (
    build_z_gross_from_tyndp_pickles,
    consumption_mix_from_tyndp_pickles,
    consumption_mix_from_z_gross,
)

In [ ]:
data_dir = Path("../data/tyndp")

production_pickle = data_dir / "DE2050_CY2009_prod.pkl"
trade_pickle = data_dir / "DE2050_CY2009_trade.pkl"
technology_mapping = data_dir / "tyndp_ei_mapping.xlsx"
country_mapping = data_dir / "tyndp_country_mapping.xlsx"

for path in [production_pickle, trade_pickle, technology_mapping, country_mapping]:
    assert path.exists(), path

## Build `Z_gross`

`Z_gross` is the gross hourly supply table with production and trade flows in one column structure.

In [ ]:
Z_gross = build_z_gross_from_tyndp_pickles(
    production_pickle=production_pickle,
    trade_pickle=trade_pickle,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    verbose=True,
)

Z_gross.shape, Z_gross.columns.names, Z_gross.index[:3]

In [ ]:
assert len(Z_gross) == 8736
assert Z_gross.columns.names == ["type", "country from", "country to", "source"]
assert {"production mix", "trade"}.issubset(Z_gross.columns.get_level_values("type"))

Z_gross.head(2)

## Compute Consumption Mix From `Z_gross`

The debug output includes the direct production/trade coefficient matrices used for the conservation checks.

In [ ]:
consumption_mix_xr, debug = consumption_mix_from_z_gross(
    Z_gross,
    check=True,
    return_debug=True,
    verbose=True,
)

consumption_mix_xr

In [ ]:
direct_total = debug["direct_total"]

direct_total.min(), direct_total.max()

In [ ]:
mix_total = consumption_mix_xr.sum(["source_country", "technology"])

float(mix_total.min()), float(mix_total.max())

In [ ]:
np.testing.assert_allclose(direct_total, 1, atol=1e-8)
np.testing.assert_allclose(mix_total, 1, atol=1e-8)

print("All conservation checks passed.")

## One-Step Pipeline

The convenience wrapper should give the same result as building `Z_gross` explicitly first.

In [ ]:
consumption_mix_direct = consumption_mix_from_tyndp_pickles(
    production_pickle=production_pickle,
    trade_pickle=trade_pickle,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    check=True,
    verbose=True,
)

xr.testing.assert_allclose(consumption_mix_direct, consumption_mix_xr)

consumption_mix_direct.shape

## Inspect One Country And Hour

In [ ]:
consumer_country = "DE"
time = consumption_mix_xr.time.values[0]

top_sources = (
    consumption_mix_xr.sel(consumer_country=consumer_country, time=time)
    .to_dataframe(name="share")
    .sort_values("share", ascending=False)
    .head(20)
)

top_sources